# Notebook 02 -- Average Curves

This notebook reproduces the average harmonic/conformal degree curves and
per-network grids from the paper "Harmonic morphisms and dynamical invariants
in network renormalization."

**Figures produced:**
- **Figure 2d:** Average harmonic degree vs compression for Laplacian, Geometric, and GNN renormalization.
- **Figure 2e:** Average conformal degree vs compression for the same three methods.
- **Figure 7:** Per-network harmonic degree curves (4-column grid).
- **Figure 8:** Per-network conformal degree curves (4-column grid).

In [ ]:
%matplotlib inline
from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

DATA_DIR = Path("../data/precomputed")

SMALL_SIZE = 10; MEDIUM_SIZE = 12; BIGGER_SIZE = 13
plt.rc("font", size=SMALL_SIZE)
plt.rc("axes", titlesize=SMALL_SIZE)
plt.rc("axes", labelsize=MEDIUM_SIZE)
plt.rc("xtick", labelsize=SMALL_SIZE)
plt.rc("ytick", labelsize=SMALL_SIZE)
plt.rc("legend", fontsize=SMALL_SIZE)
plt.rc("figure", titlesize=BIGGER_SIZE)

## Load precomputed harmonic measures

Each pickle contains Laplacian, Geometric, and GNN renormalization results for
one network. We extract compression, modified harmonic degree, and modified
conformal degree for each method.

In [ ]:
model_names = ["Laplacian", "Geometric", "GNN"]
model_colors = ["goldenrod", "salmon", "lightseagreen"]

# Column indices within the reshaped GNN DATA array (n_runs x 6):
#   0: deg_h,  1: M_deg_h,  2: std_h,
#   3: deg_cf, 4: M_deg_cf, 5: std_cf
GNN_COL_H = 1   # Modified harmonic degree
GNN_COL_CF = 4  # Modified conformal degree

compressions = {m: {} for m in model_names}
h_deg = {m: {} for m in model_names}
c_deg = {m: {} for m in model_names}
net_list = []

for pkl_path in sorted(DATA_DIR.glob("*.pkl")):
    name = pkl_path.name
    net_list.append(name)

    with open(pkl_path, "rb") as f:
        results = pickle.load(f)

    n_nodes = len(results["Graph"])

    # --- Laplacian ---
    compressions["Laplacian"][name] = np.asarray(results["Laplacian compression list"])
    h_deg["Laplacian"][name] = np.asarray(results["Laplacian Harmonic Modified"])
    c_deg["Laplacian"][name] = np.asarray(results["Laplacian Conformal Modified"])

    # --- Geometric ---
    compressions["Geometric"][name] = 1 - np.array(results["Geometric graph length"]) / n_nodes
    h_deg["Geometric"][name] = np.asarray(results["Geometric Harmonic Modified"])
    c_deg["Geometric"][name] = np.asarray(results["Geometric Conformal Modified"])

    # --- GNN ---
    compressions["GNN"][name] = 1 - np.array(results["GNN realn number of nodes"]) / n_nodes
    gnn_flat = np.array(results["GNN DATA"]).reshape(-1, 6)
    h_deg["GNN"][name] = gnn_flat[:, GNN_COL_H]
    c_deg["GNN"][name] = gnn_flat[:, GNN_COL_CF]

print(f"Loaded {len(net_list)} networks: {[n.replace('_results.pkl', '') for n in net_list]}")

## Interpolation to a common compression axis

Each method and network has a different set of compression values. To compute
averages, we interpolate all curves onto a shared axis from 0 to 1.

In [ ]:
compression_axis = np.linspace(0, 1, 100, endpoint=True)


def interpolate_measure(compressions, values, model_names, net_list):
    """Interpolate a measure onto the common compression axis.

    For GNN data the compression values are unsorted (multiple runs per
    coarsening step), so we sort before interpolation.

    Returns dicts of arrays (n_networks x 100) for each model, plus
    corresponding mean and std arrays.
    """
    interp_data = {m: [] for m in model_names}

    for model in model_names:
        for net in net_list:
            comp = compressions[model][net]
            vals = values[model][net]

            if model == "GNN":
                sort_idx = np.argsort(comp)
                comp = comp[sort_idx]
                vals = vals[sort_idx]

            f_interp = interp1d(comp, vals, kind="linear", fill_value="extrapolate")
            interp_data[model].append(f_interp(compression_axis))

    interp_arrays = {m: np.array(interp_data[m]) for m in model_names}
    means = {m: interp_arrays[m].mean(axis=0) for m in model_names}
    stds = {m: interp_arrays[m].std(axis=0) for m in model_names}
    return interp_arrays, means, stds


h_interp, h_means, h_stds = interpolate_measure(compressions, h_deg, model_names, net_list)
c_interp, c_means, c_stds = interpolate_measure(compressions, c_deg, model_names, net_list)

print("Interpolation done. Shape per model:", {m: h_interp[m].shape for m in model_names})

## Figure 2d: Average harmonic degree vs compression

Mean harmonic degree across all 16 networks, with shaded +/- 1 std band.
Geometric renormalization is plotted only from the compression-axis midpoint
onward because it begins at roughly 50% compression.

In [ ]:
def plot_average_curve(compression_axis, means, stds, model_names, model_colors,
                       ylabel, title):
    """Plot average measure vs compression with mean +/- std shading."""
    mid = len(compression_axis) // 2

    fig, ax = plt.subplots(figsize=(7, 3.5))

    for i, model in enumerate(model_names):
        if model == "Geometric":
            ax.plot(compression_axis[mid:], means[model][mid:],
                    "-o", color=model_colors[i], label=model,
                    linewidth=2, markersize=3)
            ax.fill_between(compression_axis[mid:],
                            means[model][mid:] - stds[model][mid:],
                            means[model][mid:] + stds[model][mid:],
                            color=model_colors[i], alpha=0.2)
        else:
            ax.plot(compression_axis, means[model],
                    "-o", color=model_colors[i], label=model,
                    linewidth=2, markersize=3)
            ax.fill_between(compression_axis,
                            means[model] - stds[model],
                            means[model] + stds[model],
                            color=model_colors[i], alpha=0.2)

    ax.axvline(0.5, color="gray", linestyle="dashed", alpha=0.2)
    ax.set_xlabel("Compression")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_ylim([-0.05, 1.05])
    ax.legend(ncols=1, bbox_to_anchor=(1, 0.4))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()


plot_average_curve(compression_axis, h_means, h_stds, model_names, model_colors,
                   ylabel="Harmonic Degree",
                   title="Average Harmonic Degree vs Compression")

## Figure 2e: Average conformal degree vs compression

In [ ]:
plot_average_curve(compression_axis, c_means, c_stds, model_names, model_colors,
                   ylabel="Conformal Degree",
                   title="Average Conformal Degree vs Compression")

## Figure 7: Per-network harmonic degree curves

Each panel shows the harmonic degree vs compression for one network, with all
three renormalization methods overlaid.

In [ ]:
def plot_per_network_grid(compressions, measure, net_list, model_names,
                          model_colors, ylabel, suptitle):
    """Plot a 4-column grid with one panel per network."""
    n_nets = len(net_list)
    n_cols = 4
    n_rows = int(np.ceil(n_nets / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.5 * n_rows),
                             sharex=True, sharey=True)
    axes = axes.flatten()

    for i, net in enumerate(net_list):
        ax = axes[i]
        label = net.replace("_results.pkl", "")

        # Plot order: Laplacian first, then GNN, then Geometric (on top)
        ax.plot(compressions["Laplacian"][net], measure["Laplacian"][net],
                "-o", color=model_colors[0], markersize=3, linewidth=2)
        ax.plot(compressions["GNN"][net], measure["GNN"][net],
                "-o", color=model_colors[2], markersize=3, linewidth=2)
        ax.plot(compressions["Geometric"][net], measure["Geometric"][net],
                "-o", color=model_colors[1], markersize=3, linewidth=2)

        ax.set_ylim([-0.05, 1.05])
        ax.set_title(label)
        ax.axvline(0.5, color="gray", linestyle="dashed", alpha=0.2)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if i % n_cols == 0:
            ax.set_ylabel(ylabel)
        if i >= n_nets - n_cols:
            ax.set_xlabel("Compression")

    # Hide unused axes
    for j in range(n_nets, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(suptitle, fontsize=BIGGER_SIZE, y=1.01)
    plt.tight_layout()
    plt.show()


plot_per_network_grid(compressions, h_deg, net_list, model_names, model_colors,
                      ylabel="Harmonic Degree",
                      suptitle="Per-network Harmonic Degree vs Compression")

## Figure 8: Per-network conformal degree curves

In [ ]:
plot_per_network_grid(compressions, c_deg, net_list, model_names, model_colors,
                      ylabel="Conformal Degree",
                      suptitle="Per-network Conformal Degree vs Compression")